# 🛡️ 귀갓길 안전 점수(Safety Score) 백테스팅 및 가중치 검증

이 오프라인 백테스팅 노트북은 **외부 TMap API 호출을 일절 수행하지 않고**, 로컬 데이터베이스/파일 자원(안전 시설 CSV, 안심귀갓길 SHP, 경로 캐시 JSON)만을 활용하여 안전 점수 산출 알고리즘 및 유효성을 검증합니다.

---

In [ ]:
# [1단계] 순수 Python 런타임 기반 필수 의존 패키지 자동 검사 및 동적 설치
import sys
import subprocess

required_packages = {
    "requests": "requests",
    "shapefile": "pyshp",
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn"
}

for mod_name, pkg_name in required_packages.items():
    try:
        __import__(mod_name)
    except ImportError:
        print(f"📦 [{pkg_name}] 패키지 설치 진행 중...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--break-system-packages", pkg_name])

import os
import pandas as pd
import numpy as np

# 분석용 로컬 Python 스크립트 모듈 불러오기
from facilities import load_facilities
from load_safe_routes import load_routes
from comparison_routes import load_cache
from coverage import compute_breakdown
from run_analysis import analyze

print("✅ 오프라인 백테스팅 환경 및 모듈 임포트 완료")

## 📌 Step 1: 로컬 안전 시설 및 경로 데이터셋 로딩

- **CCTV / 가로등 / 파출소**: `backend/src/main/resources/public_data/safety_facility_normalized_v2.csv`
- **안심귀갓길 경로 (Positive, Label=1)**: `backend/src/main/resources/safety_data/안심귀갓길 경로 데이터_수정.shp`
- **일반 비교군 경로 (Negative, Label=0)**: `output/comparison_routes_cache.json` (API 호출 없는 로컬 캐시)

In [ ]:
# [2단계] 로컬 자원 기반 데이터 읽기 (TMap API 호출 없음)
FACILITY_CSV = "../backend/src/main/resources/public_data/safety_facility_normalized_v2.csv"
SAFE_ROUTE_SHP = "../backend/src/main/resources/safety_data/안심귀갓길 경로 데이터_수정.shp"

print("🔍 1) 안전 시설 데이터 로딩 (CCTV, 가로등, 파출소)...")
facilities = load_facilities(FACILITY_CSV)
for f_type, f_df in facilities.items():
    print(f"   - {f_type}: {len(f_df):,}개")

print("\n🔍 2) 실제 안심귀갓길 경로 로딩 (Label=1)... ")
safe_routes = load_routes(SAFE_ROUTE_SHP)
print(f"   - 안심귀갓길 경로: {len(safe_routes)}개")

print("\n🔍 3) 일반 비교군 경로 로딩 (Label=0, 오프라인 캐시)... ")
comparison_routes = load_cache()
print(f"   - 비교군 경로: {len(comparison_routes)}개 (외부 API 요청 없음)")

## 📌 Step 2: 경로별 안전 지표 Feature Table 추출 및 점수 산출

- CCTV/가로등 커버리지 비율 (`cctv_coverage_pct`, `street_light_coverage_pct`)
- 시설 평균 간격 (`cctv_average_gap_m`, `street_light_average_gap_m`)
- 파출소 접근성 (`has_police_station`, `nearest_police_distance_m`)
- 최종 안전 점수 (`safety_score` 0~100점)

In [ ]:
# [3단계] 로컬 공간 기하 연산 기반 Feature 데이터프레임 구축
rows = []

# (1) 안심귀갓길 (Label = 1)
for r in safe_routes:
    b = compute_breakdown(
        r["points"], 
        facilities["CCTV"], 
        facilities["STREET_LIGHT"], 
        facilities["POLICE"]
    )
    rows.append({"route_id": r["route_id"], "label": 1, "gu": r["gu"], **b})

# (2) 일반 비교군 (Label = 0)
for r in comparison_routes:
    b = compute_breakdown(
        r["points"], 
        facilities["CCTV"], 
        facilities["STREET_LIGHT"], 
        facilities["POLICE"],
        route_distance_m=r.get("distance_m")
    )
    rows.append({"route_id": r["route_id"], "label": 0, "gu": "비교군", **b})

df = pd.DataFrame(rows)

# 추출된 데이터 저장
os.makedirs("output", exist_ok=True)
df.to_csv("output/features.csv", index=False)
print(f"✅ 총 {len(df)}개 경로에 대한 오프라인 Feature 추출 완료! (저장: output/features.csv)")
df.head()

## 📌 Step 3: 백테스팅 분석 및 머신러닝 변수 기여도 평가

안심귀갓길(1) 그룹과 일반비교군(0) 그룹 간의 안전 점수 및 지표 평균 비교, 상관계수, 로지스틱 회귀 모델 계수를 통해 알고리즘을 백테스팅합니다.

In [ ]:
# [4단계] 백테스팅 리포트 출력 및 결과 저장
report = analyze(df)
print(report)

# 리포트 저장
with open("output/report.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("\n✅ 백테스팅 리포트 파일 생성 완료 (output/report.txt)")

## 📌 Step 4: 그룹별 대표 지표 요약 집계 및 서울시 25개 자치구 분석

In [ ]:
# [5단계] 그룹별 대표 지표 요약
summary = df.groupby("label")[
    ["safety_score", "cctv_coverage_pct", "street_light_coverage_pct", "cctv_count", "street_light_count"]
].mean().round(2)

summary.index = ["일반 비교군 (Label=0)", "안심귀갓길 (Label=1)"]
print("📊 [오프라인 백테스팅 요약 결과]")
summary

## 📌 Step 5: 서울시 25개 자치구 중 조건을 충족하는 자치구 명단 및 개수 검증

In [ ]:
# [6단계] 서울시 25개 전체 자치구 대비 안심귀갓길 조건 충족 자치구 집계
SEOUL_25_GUS = [
    "강남구", "강동구", "강북구", "강서구", "관악구", 
    "광진구", "구로구", "금천구", "노원구", "도봉구", 
    "동대문구", "동작구", "마포구", "서대문구", "서초구", 
    "성동구", "성북구", "송파구", "양천구", "영등포구", 
    "용산구", "은평구", "종로구", "중구", "중랑구"
]

# 실제 안심귀갓길 경로 데이터(Label=1)가 존재하는 자치구 명단 추출
valid_safe_df = df[(df["label"] == 1) & (df["gu"].notna())]
satisfied_gus = sorted(list(valid_safe_df["gu"].unique()))

print(f"📊 [서울시 25개 자치구 중 조건 충족 현황]")
print(f"✅ 조건 충족 자치구 개수: 총 {len(satisfied_gus)}개 / 25개 자치구")
print(f"📋 충족 자치구 명단 ({len(satisfied_gus)}개): {', '.join(satisfied_gus)}")

# 25개 중 미포함 자치구 확인
missing_gus = [gu for gu in SEOUL_25_GUS if gu not in satisfied_gus]
if missing_gus:
    print(f"\n⚠️ 미포함 자치구 ({len(missing_gus)}개): {', '.join(missing_gus)}")